In [1]:
!pip install transformers torch accelerate plotly

In [2]:
pip install git+https://github.com/davidbau/baukit

  Cloning https://github.com/davidbau/baukit to /tmp/pip-req-build-2ju8500d
  Running command git clone --filter=blob:none --quiet https://github.com/davidbau/baukit /tmp/pip-req-build-2ju8500d
  Resolved https://github.com/davidbau/baukit to commit 9d51abd51ebf29769aecc38c4cbef459b731a36e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for baukit: filename=baukit-0.0.1-py3-none-any.whl size=59677 sha256=1f3fb7e3b14e05ffc0d5ef061d640ad11990d085170e269be511ca0f6ef8b9f3
  Stored in directory: /tmp/pip-ephem-wheel-cache-jt3r4gqq/wheels/be/54/44/c9d16a14648d7dcfaf731a41e90b28ff4014985fbc7bbc18bb
Successfully built baukit


In [3]:
# Install Hugging Face Hub
!pip install -q huggingface_hub

# Login (you'll be prompted to paste your HF token)
from huggingface_hub import notebook_login

notebook_login()


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from baukit import TraceDict
import json
from tqdm import tqdm

# ✅ Load latest Gemma-3-1B model
MODEL_NAME = "google/gemma-3-1b-it"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    output_attentions=True,
    output_hidden_states=True,
    device_map="auto",
    torch_dtype=torch.float16
)
model.eval()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['output_attentions', 'output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Gemma3ForCausalLM(
  (model): Gemma3TextModel(
    (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 1152, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma3DecoderLayer(
        (self_attn): Gemma3Attention(
          (q_proj): Linear(in_features=1152, out_features=1024, bias=False)
          (k_proj): Linear(in_features=1152, out_features=256, bias=False)
          (v_proj): Linear(in_features=1152, out_features=256, bias=False)
          (o_proj): Linear(in_features=1024, out_features=1152, bias=False)
          (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
          (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
        )
        (mlp): Gemma3MLP(
          (gate_proj): Linear(in_features=1152, out_features=6912, bias=False)
          (up_proj): Linear(in_features=1152, out_features=6912, bias=False)
          (down_proj): Linear(in_features=6912, out_features=1152, bias=False)
          (act_fn): PytorchGELUTanh()
        )
        (input_layernorm): Gemma3RMSNorm((11

In [5]:
# Pick layers to probe (out of total 26 for Gemma-3-1b)
PROBE_LAYERS = [0, 5, 10, 15, 20, 25]

def run_activation_capture(sentence, lang, emotion, exp_id):
    inputs = tokenizer(sentence, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True, output_attentions=True)

    # Hidden states shape: (layers+1, batch, tokens, hidden_dim)
    hidden_states = outputs.hidden_states
    attentions = outputs.attentions  # list of (batch, heads, tokens, tokens)

    # Collect data
    record = {
        "sentence": sentence,
        "tokens": tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]),
        "layers": {},
        "attentions": {}
    }

    # Residual activations
    for layer in PROBE_LAYERS:
        record["layers"][f"layer_{layer}"] = hidden_states[layer][0].cpu().tolist()

    # Attention maps
    for l, attn in enumerate(attentions):
        if l in PROBE_LAYERS:
            for h in range(attn.size(1)):  # num_heads
                key = f"layer{l}_head{h}"
                record["attentions"][key] = attn[0, h].cpu().tolist()

    return {
        "experiment_id": exp_id,
        "emotion": emotion,
        "language": lang,
        "data": record
    }


In [8]:
import pandas as pd

# Replace with the path to your CSV in Colab (upload or mount drive first)
df = pd.read_csv("dataset.csv")

# Convert to list of dicts (same structure we used before)
dataset = df.to_dict(orient="records")

print("Loaded", len(dataset), "rows")
print(dataset[0])  # preview first row


Loaded 320 rows
{'id': 'p001', 'emotion': 'joy', 'en': 'I am so incredibly happy right now!', 'kn': 'ನಾನು ಈಗ ನಂಬಲಾಗದಷ್ಟು ಸಂತೋಷವಾಗಿದ್ದೇನೆ!', 'mr': 'मी आता अविश्वसनीयपणे आनंदी आहे!', 'tTa': 'நான் இப்போது நம்பமுடியாத அளவிற்கு மகிழ்ச்சியாக இருக்கிறேன்!'}


In [9]:
results = []

for row in tqdm(dataset):
    for lang in ["en", "kn", "mr", "tTa"]:
        sentence = row[lang]
        out = run_activation_capture(
            sentence,
            lang=lang,
            emotion=row["emotion"],
            exp_id=row["id"]
        )
        results.append(out)

print(f"✅ Captured {len(results)} records")

# =============================
# STEP 5: Save to JSON
# =============================
output_file = "gemma3_emotion_activations.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"📂 Saved activation dataset to {output_file}")


100%|██████████| 320/320 [01:37<00:00,  3.27it/s]


✅ Captured 1280 records
📂 Saved activation dataset to gemma3_emotion_activations.json


In [10]:
!pip install numpy scipy scikit-learn matplotlib seaborn nltk ijson --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.3/148.3 kB 7.5 MB/s eta 0:00:00


In [11]:
# ================================================
# Step 1.3: Direct Cross-Language Circuit Comparison
# ================================================
# This extends your working similarity code with head-level analysis
# and circuit transfer matching.
# ================================================

!pip install numpy scipy scikit-learn matplotlib seaborn nltk ijson --quiet

import ijson
import os
from collections import defaultdict
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
nltk.download('punkt', quiet=True)

# -----------------------
# Config
# -----------------------
ACTIVATIONS_FILE = "gemma3_emotion_activations.json"
PROBE_LAYERS = [0, 5, 10, 15, 20, 25]
LANGS = ["en", "kn", "mr", "ta"]
TOP_K_CIRCUITS = 5
OUTPUT_DIR = "analysis_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# -----------------------
# Helpers
# -----------------------
def mean_pool(arr):
    arr = np.array(arr)
    return arr.mean(axis=0) if arr.ndim > 1 else arr

def cosine(a, b):
    return float(cosine_similarity(a.reshape(1, -1), b.reshape(1, -1))[0, 0])

# -----------------------
# Step 1.3A: Mean activation per emotion × layer × lang
# -----------------------
activation_sums = defaultdict(lambda: defaultdict(lambda: defaultdict(lambda: None)))
activation_counts = defaultdict(lambda: defaultdict(lambda: defaultdict(int)))

print("Streaming activations file:", ACTIVATIONS_FILE)
with open(ACTIVATIONS_FILE, "r", encoding="utf-8") as f:
    parser = ijson.items(f, "item")
    for obj in parser:
        emo, lang = obj["emotion"], obj["language"]
        for layer in PROBE_LAYERS:
            layer_key = f"layer_{layer}"
            if layer_key not in obj["data"]["layers"]:
                continue
            pooled = mean_pool(obj["data"]["layers"][layer_key])
            if activation_sums[emo][layer][lang] is None:
                activation_sums[emo][layer][lang] = pooled
            else:
                activation_sums[emo][layer][lang] += pooled
            activation_counts[emo][layer][lang] += 1

# Compute means
activation_means = defaultdict(lambda: defaultdict(dict))
for emo in activation_sums:
    for layer in PROBE_LAYERS:
        for lang in LANGS:
            if activation_counts[emo][layer][lang] > 0:
                activation_means[emo][layer][lang] = activation_sums[emo][layer][lang] / activation_counts[emo][layer][lang]
            else:
                activation_means[emo][layer][lang] = None

# Heatmaps
for emo in activation_means:
    for layer in PROBE_LAYERS:
        sims = np.zeros((len(LANGS), len(LANGS)))
        for i, la in enumerate(LANGS):
            for j, lb in enumerate(LANGS):
                v1, v2 = activation_means[emo][layer][la], activation_means[emo][layer][lb]
                sims[i, j] = cosine(v1, v2) if v1 is not None and v2 is not None else np.nan
        plt.figure(figsize=(5, 4))
        sns.heatmap(sims, xticklabels=LANGS, yticklabels=LANGS,
                    vmin=0, vmax=1, annot=True, fmt=".2f",
                    cmap="viridis", mask=np.isnan(sims))
        plt.title(f"Emotion={emo}  Layer={layer}")
        fname = os.path.join(OUTPUT_DIR, f"sim_heatmap_{emo}_layer{layer}.png")
        plt.tight_layout()
        plt.savefig(fname, dpi=150)
        plt.close()
        print(f"Saved heatmap: {fname}")

# -----------------------
# Step 1.3B: Attention Head Analysis
# -----------------------
# Collect per-head mean activations
head_sums = defaultdict(lambda: defaultdict(lambda: defaultdict(lambda: defaultdict(lambda: None))))
head_counts = defaultdict(lambda: defaultdict(lambda: defaultdict(lambda: defaultdict(int))))

with open(ACTIVATIONS_FILE, "r", encoding="utf-8") as f:
    parser = ijson.items(f, "item")
    for obj in parser:
        emo, lang = obj["emotion"], obj["language"]
        if "heads" not in obj["data"]:
            continue  # Skip if head data missing
        for layer_key, head_dict in obj["data"]["heads"].items():
            layer = int(layer_key.split("_")[1])
            if layer not in PROBE_LAYERS:
                continue
            for head_idx, head_act in head_dict.items():
                pooled = mean_pool(head_act)
                if head_sums[emo][layer][lang][head_idx] is None:
                    head_sums[emo][layer][lang][head_idx] = pooled
                else:
                    head_sums[emo][layer][lang][head_idx] += pooled
                head_counts[emo][layer][lang][head_idx] += 1

# Compute head means
head_means = defaultdict(lambda: defaultdict(lambda: defaultdict(dict)))
for emo in head_sums:
    for layer in PROBE_LAYERS:
        for lang in LANGS:
            for head_idx in head_sums[emo][layer][lang]:
                if head_counts[emo][layer][lang][head_idx] > 0:
                    head_means[emo][layer][lang][head_idx] = (
                        head_sums[emo][layer][lang][head_idx] / head_counts[emo][layer][lang][head_idx]
                    )

# -----------------------
# Step 1.3C: Circuit Transfer Matrix
# -----------------------
for emo in head_means:
    print(f"\n=== Circuit Transfer for Emotion: {emo} ===")
    for layer in PROBE_LAYERS:
        if "en" not in head_means[emo][layer]:
            continue
        # Find top-K heads in English
        english_heads = []
        for h, vec in head_means[emo][layer]["en"].items():
            english_heads.append((h, np.linalg.norm(vec)))
        top_heads = sorted(english_heads, key=lambda x: -x[1])[:TOP_K_CIRCUITS]

        print(f"Layer {layer} Top-{TOP_K_CIRCUITS} English heads: {top_heads}")

        # Check matches in other languages
        for lang in LANGS:
            if lang == "en":
                continue
            matches = 0
            for h, _ in top_heads:
                if h in head_means[emo][layer][lang]:
                    matches += 1
            success_rate = matches / TOP_K_CIRCUITS
            print(f"  English → {lang}: {matches}/{TOP_K_CIRCUITS} circuits match ({success_rate:.0%})")



Streaming activations file: gemma3_emotion_activations.json
Saved heatmap: analysis_outputs/sim_heatmap_joy_layer0.png
Saved heatmap: analysis_outputs/sim_heatmap_joy_layer5.png
Saved heatmap: analysis_outputs/sim_heatmap_joy_layer10.png
Saved heatmap: analysis_outputs/sim_heatmap_joy_layer15.png
Saved heatmap: analysis_outputs/sim_heatmap_joy_layer20.png
Saved heatmap: analysis_outputs/sim_heatmap_joy_layer25.png
Saved heatmap: analysis_outputs/sim_heatmap_trust_layer0.png
Saved heatmap: analysis_outputs/sim_heatmap_trust_layer5.png
Saved heatmap: analysis_outputs/sim_heatmap_trust_layer10.png
Saved heatmap: analysis_outputs/sim_heatmap_trust_layer15.png
Saved heatmap: analysis_outputs/sim_heatmap_trust_layer20.png
Saved heatmap: analysis_outputs/sim_heatmap_trust_layer25.png
Saved heatmap: analysis_outputs/sim_heatmap_fear_layer0.png
Saved heatmap: analysis_outputs/sim_heatmap_fear_layer5.png
Saved heatmap: analysis_outputs/sim_heatmap_fear_layer10.png
Saved heatmap: analysis_outputs